# Class Lab: getting started with PySpark

# First steps in PySpark

In [2]:
from pyspark.sql import SparkSession

# create a Spark session
spark = SparkSession.builder.appName("PySpark_Get_Started").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/11 13:52:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# shut down the current active SparkSession
spark.stop()

# Leverage RDD (Resilient Distributed Dataset)

## RDD creation

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDD_demo").getOrCreate()
numbers = [1, 2, 3, 4, 5]
# create a RDD from a list
rdd = spark.sparkContext.parallelize(numbers) 

# collect action: retrieve all elements of the RDD
rdd.collect()

[1, 2, 3, 4, 5]

In [5]:
# create RDD from a list of tuples
fruits = [("apple", 5), ("kiwi", 1), ("banana", 3), ("apple", 3), ("kiwi", 7)]
fruits_rdd = spark.sparkContext.parallelize(fruits)
print("All elements of the fruits rdd", fruits_rdd.collect())


All elements of the fruits rdd [('apple', 5), ('kiwi', 1), ('banana', 3), ('apple', 3), ('kiwi', 7)]


## RDD operations: Actions

In [6]:
# Count action: count the number of elements in the RDD
fruits_rdd.count()

5

In [7]:
# First action: retrieve the first element of the RDD
fruits_rdd.first()

('apple', 5)

In [8]:
# Take action: retrieve the first n elements of the RDD
fruits_rdd.take(2)


[('apple', 5), ('kiwi', 1)]

In [9]:
# Foreach action: apply a function to each element of the RDD
fruits_rdd.foreach(lambda x: print(x))


('banana', 3)
('apple', 3)
('kiwi', 1)
('apple', 5)
('kiwi', 7)


## RDD operations: Transformations

In [10]:
# Map transformation: apply a function to each element of the RDD
mapped_fruits_rdd = fruits_rdd.map(lambda x: (x[0].upper(), x[1]))
mapped_fruits_rdd.collect()

[('APPLE', 5), ('KIWI', 1), ('BANANA', 3), ('APPLE', 3), ('KIWI', 7)]

In [11]:
# Filter transformation: select elements that meet a certain condition
filtered_fruits_rdd = fruits_rdd.filter(lambda x: x[1] >= 5)
filtered_fruits_rdd.collect()

[('apple', 5), ('kiwi', 7)]

In [12]:
# ReduceByKey transformation: aggregate values by key
reduced_fruits_rdd = fruits_rdd.reduceByKey(lambda x, y: x + y)
reduced_fruits_rdd.collect()

[('kiwi', 8), ('apple', 8), ('banana', 3)]

In [13]:
# SortBy transformation: sort the RDD by key
sorted_fruits_rdd = fruits_rdd.sortBy(lambda x: x[1], ascending=False)
sorted_fruits_rdd.collect()

[('kiwi', 7), ('apple', 5), ('banana', 3), ('apple', 3), ('kiwi', 1)]

## Read / Write RDDs from  / to text file

In [14]:
# Save action: write the RDD to a text file
fruits_rdd.saveAsTextFile("../txt/fruits.txt")

In [15]:
# Create RDD from a text file
rdd = spark.sparkContext.textFile("../txt/fruits.txt")
rdd.collect()

["('kiwi', 1)", "('banana', 3)", "('apple', 5)", "('apple', 3)", "('kiwi', 7)"]

In [16]:
spark.stop()  # shut down the current active SparkSession

# Leverage DataFrame

In [17]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataFrame_demo").getOrCreate()

# Create a DataFrame
data = [("apple", 5), ("kiwi", 1), ("banana", 3), ("apple", 3), ("kiwi", 7)]
df = spark.createDataFrame(data, ["Fruit", "Amount"])
# df.show() # text mode table
pretty_df = df.toPandas() # well-formatted data frame by pandas style
pretty_df

,Fruit,Amount
0,apple,5
1,kiwi,1
2,banana,3
3,apple,3
4,kiwi,7


In [18]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataFrame_demo").getOrCreate()

In [19]:
# load data into DataFrame

data_file_path = "../txt/stocks.txt"
df = spark.read.csv(data_file_path, header=True, inferSchema=True)

# Show the first few rows of the DataFrame
df.printSchema()
df.show(10)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)

+---+----------------+-----------+--------+-------+
| id|            name|   category|quantity|  price|
+---+----------------+-----------+--------+-------+
|  1|          iPhone|Electronics|      10| 899.99|
|  2|         Macbook|Electronics|       5|1299.99|
|  3|            iPad|Electronics|      15| 499.99|
|  4|      Samsung TV|Electronics|       8| 799.99|
|  5|           LG TV|Electronics|      10| 699.99|
|  6|      Nike Shoes|   Clothing|      30|  99.99|
|  7|    Adidas Shoes|   Clothing|      25|  89.99|
|  8| Sony Headphones|Electronics|      12| 149.99|
|  9|Beats Headphones|Electronics|      20| 199.99|
| 10|    Dining Table|  Furniture|      10| 249.99|
+---+----------------+-----------+--------+-------+
only showing top 10 rows


In [20]:
# select specific columns
selected_columns = df.select("id", "name", "price")
selected_columns.show(10)

+---+----------------+-------+
| id|            name|  price|
+---+----------------+-------+
|  1|          iPhone| 899.99|
|  2|         Macbook|1299.99|
|  3|            iPad| 499.99|
|  4|      Samsung TV| 799.99|
|  5|           LG TV| 699.99|
|  6|      Nike Shoes|  99.99|
|  7|    Adidas Shoes|  89.99|
|  8| Sony Headphones| 149.99|
|  9|Beats Headphones| 199.99|
| 10|    Dining Table| 249.99|
+---+----------------+-------+
only showing top 10 rows


In [21]:
# filter rows based on a condition
filtered_df = df.filter(df["price"] > 100)
filtered_df.show(10)

+---+----------------+-----------+--------+-------+
| id|            name|   category|quantity|  price|
+---+----------------+-----------+--------+-------+
|  1|          iPhone|Electronics|      10| 899.99|
|  2|         Macbook|Electronics|       5|1299.99|
|  3|            iPad|Electronics|      15| 499.99|
|  4|      Samsung TV|Electronics|       8| 799.99|
|  5|           LG TV|Electronics|      10| 699.99|
|  8| Sony Headphones|Electronics|      12| 149.99|
|  9|Beats Headphones|Electronics|      20| 199.99|
| 10|    Dining Table|  Furniture|      10| 249.99|
| 11|      Study Desk|  Furniture|       8| 149.99|
| 17|  Leather Jacket|   Clothing|      15| 199.99|
+---+----------------+-----------+--------+-------+
only showing top 10 rows


In [22]:
# GroupBy and Aggregation
grouped_df = df.groupBy("category").agg({"quantity": "sum", "price": "avg"})
grouped_df.show()

+-----------+-------------+------------------+
|   category|sum(quantity)|        avg(price)|
+-----------+-------------+------------------+
|       Food|          450|2.2960000000000003|
|     Sports|           35|             34.99|
|Electronics|           98| 586.6566666666665|
|   Clothing|          200|  99.2757142857143|
|  Furniture|           41|            141.99|
|Accessories|           55|             27.49|
+-----------+-------------+------------------+



In [25]:
# Join with another DataFrame
other_data = [("Food", "1"), ("Sports", "2"), ("Electronics", "3"), ("Clothing", "4"), ("Furniture", "5"), ("Accessories", "6")]
other_columns = ["category", "typy_id"]
other_df = spark.createDataFrame(other_data, other_columns)

joined_df = df.join(other_df, df["category"] == other_df["category"], "inner")
joined_df.show()

+---+----------------+-----------+--------+-------+-----------+-------+
| id|            name|   category|quantity|  price|   category|typy_id|
+---+----------------+-----------+--------+-------+-----------+-------+
| 16|   Salmon Fillet|       Food|      30|   5.99|       Food|      1|
| 15|  Chicken Breast|       Food|      50|   3.99|       Food|      1|
| 14|         Oranges|       Food|     120|   0.75|       Food|      1|
| 13|         Bananas|       Food|     150|   0.25|       Food|      1|
| 12|          Apples|       Food|     100|    0.5|       Food|      1|
| 20|    Dumbbell Set|     Sports|      15|  49.99|     Sports|      2|
| 19|        Yoga Mat|     Sports|      20|  19.99|     Sports|      2|
| 27|         Printer|Electronics|       8| 129.99|Electronics|      3|
| 26|          Camera|Electronics|      10| 599.99|Electronics|      3|
|  9|Beats Headphones|Electronics|      20| 199.99|Electronics|      3|
|  8| Sony Headphones|Electronics|      12| 149.99|Electronics| 

In [26]:
# sort by a column
sorted_df = df.orderBy("price", ascending=False)
sorted_df.show(10)

+---+----------------+-----------+--------+-------+
| id|            name|   category|quantity|  price|
+---+----------------+-----------+--------+-------+
|  2|         Macbook|Electronics|       5|1299.99|
|  1|          iPhone|Electronics|      10| 899.99|
|  4|      Samsung TV|Electronics|       8| 799.99|
|  5|           LG TV|Electronics|      10| 699.99|
| 26|          Camera|Electronics|      10| 599.99|
|  3|            iPad|Electronics|      15| 499.99|
| 10|    Dining Table|  Furniture|      10| 249.99|
|  9|Beats Headphones|Electronics|      20| 199.99|
| 17|  Leather Jacket|   Clothing|      15| 199.99|
|  8| Sony Headphones|Electronics|      12| 149.99|
+---+----------------+-----------+--------+-------+
only showing top 10 rows


In [27]:
# sort by multiple columns
sorted_df = df.orderBy(["category", "id"], ascending=[True, False])
sorted_df.show(10)

+---+--------------+-----------+--------+------+
| id|          name|   category|quantity| price|
+---+--------------+-----------+--------+------+
| 25|      Backpack|Accessories|      30| 24.99|
| 24|    Laptop Bag|Accessories|      25| 29.99|
| 30|      Sneakers|   Clothing|      40| 79.99|
| 29|       T-shirt|   Clothing|      50| 14.99|
| 28|         Jeans|   Clothing|      30| 59.99|
| 18|   Winter Coat|   Clothing|      10|149.99|
| 17|Leather Jacket|   Clothing|      15|199.99|
|  7|  Adidas Shoes|   Clothing|      25| 89.99|
|  6|    Nike Shoes|   Clothing|      30| 99.99|
| 27|       Printer|Electronics|       8|129.99|
+---+--------------+-----------+--------+------+
only showing top 10 rows


In [28]:
# get distinct rows
distinct_rows = df.select("category").distinct()
distinct_rows.show()

+-----------+
|   category|
+-----------+
|       Food|
|     Sports|
|Electronics|
|   Clothing|
|  Furniture|
|Accessories|
+-----------+



In [29]:
# drop columns
dropped_columns = df.drop("category", "name")
dropped_columns.show(10)

+---+--------+-------+
| id|quantity|  price|
+---+--------+-------+
|  1|      10| 899.99|
|  2|       5|1299.99|
|  3|      15| 499.99|
|  4|       8| 799.99|
|  5|      10| 699.99|
|  6|      30|  99.99|
|  7|      25|  89.99|
|  8|      12| 149.99|
|  9|      20| 199.99|
| 10|      10| 249.99|
+---+--------+-------+
only showing top 10 rows


In [30]:
# add new calculated columns
df_with_new_column = df.withColumn("revenue", df["price"] * df["quantity"])
df_with_new_column.show(10)

+---+----------------+-----------+--------+-------+-------+
| id|            name|   category|quantity|  price|revenue|
+---+----------------+-----------+--------+-------+-------+
|  1|          iPhone|Electronics|      10| 899.99| 8999.9|
|  2|         Macbook|Electronics|       5|1299.99|6499.95|
|  3|            iPad|Electronics|      15| 499.99|7499.85|
|  4|      Samsung TV|Electronics|       8| 799.99|6399.92|
|  5|           LG TV|Electronics|      10| 699.99| 6999.9|
|  6|      Nike Shoes|   Clothing|      30|  99.99| 2999.7|
|  7|    Adidas Shoes|   Clothing|      25|  89.99|2249.75|
|  8| Sony Headphones|Electronics|      12| 149.99|1799.88|
|  9|Beats Headphones|Electronics|      20| 199.99| 3999.8|
| 10|    Dining Table|  Furniture|      10| 249.99| 2499.9|
+---+----------------+-----------+--------+-------+-------+
only showing top 10 rows


In [31]:
# rename columns with alias
renamed_columns = df.withColumnRenamed("id", "product_id") \
                    .withColumnRenamed("name", "product_name") \
                    .withColumnRenamed("price", "product_price")
renamed_columns.show(10)

+----------+----------------+-----------+--------+-------------+
|product_id|    product_name|   category|quantity|product_price|
+----------+----------------+-----------+--------+-------------+
|         1|          iPhone|Electronics|      10|       899.99|
|         2|         Macbook|Electronics|       5|      1299.99|
|         3|            iPad|Electronics|      15|       499.99|
|         4|      Samsung TV|Electronics|       8|       799.99|
|         5|           LG TV|Electronics|      10|       699.99|
|         6|      Nike Shoes|   Clothing|      30|        99.99|
|         7|    Adidas Shoes|   Clothing|      25|        89.99|
|         8| Sony Headphones|Electronics|      12|       149.99|
|         9|Beats Headphones|Electronics|      20|       199.99|
|        10|    Dining Table|  Furniture|      10|       249.99|
+----------+----------------+-----------+--------+-------------+
only showing top 10 rows


In [32]:
spark.stop()

# Leverage Spark SQL

In [33]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkSQL_demo").getOrCreate()

In [34]:
# load data into DataFrame
data_file_path = "../csv/persons.csv"
df = spark.read.csv(data_file_path, header=True, inferSchema=True)

# Show the first few rows of the DataFrame
df.printSchema()
df.show(10)

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+------------------+---+------+------+
|              name|age|gender|salary|
+------------------+---+------+------+
|          John Doe| 30|  Male| 50000|
|        Jane Smith| 25|Female| 45000|
|     David Johnson| 35|  Male| 60000|
|       Emily Davis| 28|Female| 52000|
|    Michael Wilson| 40|  Male| 75000|
|       Sarah Brown| 32|Female| 58000|
|        Robert Lee| 29|  Male| 51000|
|       Lisa Garcia| 27|Female| 49000|
|    James Martinez| 38|  Male| 70000|
|Jennifer Rodriguez| 26|Female| 47000|
+------------------+---+------+------+
only showing top 10 rows


In [35]:
# register DataFrame as a temporary view
df.createOrReplaceTempView("persons")

In [36]:
# write SQL queries
result = spark.sql("SELECT * FROM persons WHERE age > 30")
result.show()

+------------------+---+------+------+
|              name|age|gender|salary|
+------------------+---+------+------+
|     David Johnson| 35|  Male| 60000|
|    Michael Wilson| 40|  Male| 75000|
|       Sarah Brown| 32|Female| 58000|
|    James Martinez| 38|  Male| 70000|
|  William Anderson| 33|  Male| 62000|
|   Karen Hernandez| 31|Female| 55000|
|Christopher Taylor| 37|  Male| 69000|
|     Matthew Davis| 36|  Male| 67000|
|     Daniel Miller| 34|  Male| 64000|
|      Linda Martin| 39|Female| 71000|
+------------------+---+------+------+



In [37]:
avg_salary_by_gender = spark.sql("""
    SELECT gender, AVG(salary) as avg_salary
    FROM persons
    GROUP BY gender
""")
avg_salary_by_gender.show()

+------+----------+
|gender|avg_salary|
+------+----------+
|Female|   52300.0|
|  Male|   62100.0|
+------+----------+



In [38]:
# check if a temporary view exists
if spark.catalog.tableExists("persons"):
    print("Temporary view 'persons' exists.")
else:
    print("Temporary view 'persons' does not exist.")


Temporary view 'persons' exists.


In [39]:
# drop a temporary view
spark.catalog.dropTempView("persons")


True